# Module 4 - Topic 1: Consuming External APIs with Python
**Generative AI Fellowship — Beginner**

In this demo you will:
- Install and import the requests library
- Make GET requests to a free public API
- Read status codes and parse JSON responses
- Pass query parameters
- Handle errors gracefully

> **No API key needed** — all calls in this demo use the free REST Countries API.


In [ ]:
# Cell 1 - Install
# Run this once to install the requests library
# !pip install requests

In [ ]:
# Cell 2 - Import
import requests

print("requests version:", requests.__version__)
print("Ready to call APIs.")

In [ ]:
# Cell 3 - Your First GET Request
# We call the REST Countries API to get information about Nigeria
# This API is free and requires no authentication

response = requests.get("https://restcountries.com/v3.1/name/nigeria")

# Always check the status code first
print("Status code:", response.status_code)
# 200 means success

In [ ]:
# Cell 4 - Reading the Response
# The response object has several useful properties

print("Status code:", response.status_code)    # 200
print("URL called: ", response.url)             # the full URL
print("Content type:", response.headers["Content-Type"])  # application/json

In [ ]:
# Cell 5 - Parsing the JSON Response
# response.json() converts the JSON string into a Python list or dict

data = response.json()

# The API returns a list — we take the first result
nigeria = data[0]

# Access the fields
print("Country:", nigeria["name"]["common"])
print("Official name:", nigeria["name"]["official"])
print("Capital:", nigeria["capital"][0])
print("Population:", nigeria["population"])
print("Region:", nigeria["region"])

In [ ]:
# Cell 6 - Accessing Nested JSON
# API responses are often nested — drill down using chained keys

# Currency information (nested two levels deep)
currency_code = list(nigeria["currencies"].keys())[0]   # 'NGN'
currency_name = nigeria["currencies"][currency_code]["name"]
currency_symbol = nigeria["currencies"][currency_code]["symbol"]

print(f"Currency code: {currency_code}")
print(f"Currency name: {currency_name}")
print(f"Currency symbol: {currency_symbol}")

# Languages
print("\nLanguages:", nigeria["languages"])

# Timezones
print("Timezone:", nigeria.get("timezones", ["Unknown"])[0])

In [ ]:
# Cell 7 - Safe Access with .get()
# Use .get() with a default when a key might not always be present
# This prevents KeyError crashes when API responses vary

# Safe access — returns the default if the key is missing
population = nigeria.get("population", 0)
area = nigeria.get("area", 0)
flag_emoji = nigeria.get("flag", "N/A")

print(f"Population: {population:,}")
print(f"Area: {area:,} km²")
print(f"Flag: {flag_emoji}")

In [ ]:
# Cell 8 - Calling Multiple Countries
# Let's compare Nigeria with other African countries

countries = ["nigeria", "ghana", "kenya", "southafrica"]

for country_name in countries:
    response = requests.get(f"https://restcountries.com/v3.1/name/{country_name}")
    if response.status_code == 200:
        country = response.json()[0]
        name = country["name"]["common"]
        population = country.get("population", 0)
        capital = country.get("capital", ["N/A"])[0]
        print(f"{name} | Capital: {capital} | Population: {population:,}")

In [ ]:
# Cell 9 - Query Parameters
# Use a params dict to filter API results — never build URLs manually

# Get all countries in Africa, requesting specific fields only
params = {
    "fields": "name,capital,population,currencies"
}

response = requests.get(
    "https://restcountries.com/v3.1/region/africa",
    params=params
)

# Show the full URL that was built automatically
print("URL called:", response.url)
print("Status:", response.status_code)

african_countries = response.json()
print(f"\nNumber of African countries returned: {len(african_countries)}")

# Show the first 5
print("\nFirst 5 countries:")
for country in african_countries[:5]:
    print(" -", country["name"]["common"])

In [ ]:
# Cell 10 - Handling a 404 Error
# What happens when we request something that does not exist?

response = requests.get("https://restcountries.com/v3.1/name/abcdefghijk")

print("Status code:", response.status_code)  # 404

# Check before calling .json()
if response.status_code == 200:
    print("Found:", response.json()[0]["name"]["common"])
elif response.status_code == 404:
    print("Country not found")
else:
    print(f"Unexpected status: {response.status_code}")

In [ ]:
# Cell 11 - Error Handling with try/except
# The professional pattern for making API calls safely

def get_country_info(country_name: str) -> dict:
    """Fetch country info from REST Countries API. Returns None on failure."""
    try:
        response = requests.get(
            f"https://restcountries.com/v3.1/name/{country_name}",
            timeout=10  # always set a timeout
        )
        response.raise_for_status()  # raises exception for 4xx/5xx
        return response.json()[0]

    except requests.HTTPError:
        print(f"Country '{country_name}' not found")
        return None
    except requests.ConnectionError:
        print("No internet connection")
        return None
    except requests.Timeout:
        print("Request timed out")
        return None


# Test with a valid country
result = get_country_info("nigeria")
if result:
    print("Found:", result["name"]["common"])

# Test with an invalid country
result = get_country_info("notacountry")
print("Result:", result)

In [ ]:
# Cell 12 - Building a Nigerian Country Summary
# Put it all together — a clean summary of Nigeria from the API

nigeria_data = get_country_info("nigeria")

if nigeria_data:
    currency_code = list(nigeria_data["currencies"].keys())[0]
    currency_info = nigeria_data["currencies"][currency_code]

    print("=" * 40)
    print("NIGERIA — COUNTRY SUMMARY")
    print("=" * 40)
    print(f"Official name : {nigeria_data['name']['official']}")
    print(f"Capital       : {nigeria_data.get('capital', ['N/A'])[0]}")
    print(f"Population    : {nigeria_data.get('population', 0):,}")
    print(f"Area          : {nigeria_data.get('area', 0):,} km²")
    print(f"Currency      : {currency_info['name']} ({currency_info['symbol']})")
    print(f"Languages     : {', '.join(nigeria_data.get('languages', {}).values())}")
    print(f"Region        : {nigeria_data.get('region', 'N/A')}")
    print(f"Flag          : {nigeria_data.get('flag', '')}")